# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

### ⚡ Configuration requise dans Kaggle (panneau à droite) :
1. **Accelerator** → `GPU T4 x2` (ou `P100`)
2. **Internet** → `ON`
3. Cliquez sur **Run All** (ou exécutez les cellules 1 à 5 dans l'ordre).
4. À la fin, copiez l'URL Cloudflare affichée (`https://xxxx.trycloudflare.com`) et collez-la dans AbiArt → Backend Kaggle → Connect.

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 1 — VÉRIFICATION DU GPU
# ══════════════════════════════════════════════════
import subprocess, sys, shutil

has_gpu = False
if shutil.which('nvidia-smi'):
    try:
        r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if r.returncode == 0:
            has_gpu = True
            print(r.stdout.split('\n')[0])
            print('\u2713 GPU actif et détecté !')
    except Exception:
        pass

if not has_gpu:
    print('\u274c ERREUR CRITIQUE : Le GPU n\'est pas activé sur Kaggle !')
    print('    \u2192 Dans le volet à droite de l\'écran Kaggle :')
    print('    \u2192 Section "Session options" -> "Accelerator"')
    print('    \u2192 Changez "None" par "GPU T4 x2" (ou "GPU P100")')
    print('    \u2192 Vérifiez que "Internet" est bien sur "Internet On"')
    print('    \u2192 Cliquez sur le bouton de redémarrage de la session.')
    raise RuntimeError("Veuillez activer le GPU dans le panneau de droite avant de continuer.")

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 2 — INSTALLATION COMPLETE & REPOSITORIES AUTO-FIX
# ══════════════════════════════════════════════════
import os, sys, shutil, subprocess

WEBUI = '/kaggle/working/stable-diffusion-webui'
os.environ['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'
os.environ['GENERATIVE_MODELS_REPO'] = 'https://github.com/Stability-AI/generative-models.git'
os.environ['K_DIFFUSION_REPO'] = 'https://github.com/crowsonkb/k-diffusion.git'

# 1. Cloner Automatic1111 si absent
if not os.path.exists(WEBUI):
    print('Clonage d\'Automatic1111...')
    !git clone --depth 1 https://github.com/AUTOMATIC1111/stable-diffusion-webui.git {WEBUI}
    !rm -rf {WEBUI}/.git 2>/dev/null || true

%cd {WEBUI}

# 2. Créer sitecustomize.py pour blinder l'import de pytorch_lightning.utilities.distributed
with open(os.path.join(WEBUI, 'sitecustomize.py'), 'w') as f:
    f.write('''
import sys
try:
    import pytorch_lightning.utilities as plu
    if not hasattr(plu, 'distributed'):
        import types
        dist = types.ModuleType('pytorch_lightning.utilities.distributed')
        try:
            from pytorch_lightning.utilities.rank_zero import rank_zero_only
            dist.rank_zero_only = rank_zero_only
        except Exception:
            dist.rank_zero_only = lambda fn: fn
        plu.distributed = dist
        sys.modules['pytorch_lightning.utilities.distributed'] = dist
except Exception:
    pass
''')

# 3. Patcher requirements_versions.txt
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements_versions.txt
!sed -i 's/scikit-image==0.21.0/scikit-image>=0.21.0/g' requirements_versions.txt

# 4. Installation exhaustive des dépendances A1111 (sans cache pour préserver le disque)
print('\n[1/4] Installation de toutes les dépendances Python...')
!pip install --no-cache-dir -q \
    diskcache \
    blendmodes \
    clean-fid \
    facexlib \
    inflection \
    jsonmerge \
    resize-right \
    torchdiffeq \
    torchsde \
    tomesd \
    spandrel \
    spandrel-extra-arches \
    pillow-avif-plugin \
    piexif \
    "fastapi>=0.94.0" \
    "gradio==3.41.2" \
    "kornia>=0.6.7" \
    "lark>=1.1.2" \
    "einops>=0.4.1" \
    "omegaconf>=2.2.3" \
    "accelerate>=0.21.0" \
    "safetensors>=0.4.2" \
    "transformers>=4.30.2" \
    "open-clip-torch>=2.20.0" \
    "pytorch-lightning==1.9.4" \
    "Pillow>=10.0.0" \
    "scikit-image>=0.21.0" \
    xformers ultralytics mediapipe

print('\n[2/4] Installation de CLIP...')
!pip install --no-cache-dir -q --no-build-isolation https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip

# 5. Cloner proprement TOUS les sous-dépôts avec leurs commits exacts
print('\n[3/4] Configuration des sous-dépôts (generative-models, stablediffusion, k-diffusion)...')
os.makedirs('repositories', exist_ok=True)

repos = [
    {
        'name': 'generative-models',
        'url': 'https://github.com/Stability-AI/generative-models.git',
        'commit': '45c443b316737a4ab6e40413d7794a7f5657c19f'
    },
    {
        'name': 'stable-diffusion-stability-ai',
        'url': 'https://github.com/w-e-w/stablediffusion.git',
        'commit': 'cf1d67a6fd5ea1aa600c4df58e5b47da45f6bdbf'
    },
    {
        'name': 'k-diffusion',
        'url': 'https://github.com/crowsonkb/k-diffusion.git',
        'commit': 'ab527a9a6d347f364e3d185ba6d714e22d80cb3c'
    },
    {
        'name': 'BLIP',
        'url': 'https://github.com/salesforce/BLIP.git',
        'commit': '48211a1594f1321b00f14c9f7a5b4813144b2fb9'
    },
    {
        'name': 'CodeFormer',
        'url': 'https://github.com/sczhou/CodeFormer.git',
        'commit': 'c5b4593074ba6214284d6acd5f1719b6c5d739af'
    },
    {
        'name': 'stable-diffusion-webui-assets',
        'url': 'https://github.com/AUTOMATIC1111/stable-diffusion-webui-assets.git',
        'commit': '6f7db241d2f8ba7457bac5ca9753331f0c266917'
    }
]

for r in repos:
    dest = os.path.join(WEBUI, 'repositories', r['name'])
    need_clone = True
    if os.path.exists(dest):
        if r['commit']:
            chk = subprocess.run(['git', '-C', dest, 'checkout', r['commit']], capture_output=True, text=True)
            if chk.returncode == 0:
                print(f"  ✓ {r['name']} déjà prêt ({r['commit'][:7]})")
                need_clone = False
            else:
                print(f"  ⚠️ Nettoyage de l'ancien clone de {r['name']}...")
                shutil.rmtree(dest, ignore_errors=True)
        else:
            need_clone = False
            print(f"  ✓ {r['name']} présent")
    
    if need_clone:
        print(f"  Clonage de {r['name']}...")
        subprocess.run(['git', 'clone', r['url'], dest], check=True)
        if r['commit']:
            try:
                subprocess.run(['git', '-C', dest, 'checkout', r['commit']], check=True)
                print(f"  ✓ {r['name']} verrouillé sur {r['commit'][:7]}")
            except Exception:
                print(f"  ⚠️ {r['name']} checkout passé, utilisation de la branche principale")
    
    # Supprimer l'historique .git pour économiser 5 Go sur le disque Kaggle
    git_dir = os.path.join(dest, '.git')
    if os.path.exists(git_dir):
        shutil.rmtree(git_dir, ignore_errors=True)

# 6. Extensions utiles
print('\n[4/4] Vérification des extensions...')
!git clone --depth 1 https://github.com/Bing-su/adetailer.git extensions/adetailer 2>/dev/null || true
!git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git extensions/sd-webui-controlnet 2>/dev/null || true
!rm -rf extensions/*/.git 2>/dev/null || true

# 7. Nettoyage du cache pip
!pip cache purge 2>/dev/null || true
!rm -rf /root/.cache/pip /root/.cache/huggingface 2>/dev/null || true

# 8. Diagnostic espace disque
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'\n✓ Environnement configuré ! Espace disque disponible : {free / (1024**3):.1f} Go libres sur {total / (1024**3):.1f} Go')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 3 — AUDIT COMPLET DE TOUS LES MODULES
# ══════════════════════════════════════════════════
import importlib, sys, os

WEBUI = '/kaggle/working/stable-diffusion-webui'
if WEBUI not in sys.path:
    sys.path.insert(0, WEBUI)
for sub in ['stable-diffusion-stability-ai', 'generative-models', 'k-diffusion', 'BLIP']:
    p = os.path.join(WEBUI, 'repositories', sub)
    if p not in sys.path:
        sys.path.insert(0, p)

modules_a_tester = [
    ('torch',                          'PyTorch'),
    ('torchvision',                    'TorchVision'),
    ('xformers',                       'xFormers'),
    ('PIL',                            'Pillow'),
    ('skimage',                        'scikit-image'),
    ('diskcache',                      'diskcache (A1111 Cache)'),
    ('clip',                           'CLIP (OpenAI)'),
    ('open_clip',                      'open_clip (SDXL Conditioning)'),
    ('pytorch_lightning',              'PyTorch Lightning'),
    ('pytorch_lightning.utilities.distributed', 'PL distributed (rank_zero_only)'),
    ('gradio',                         'Gradio'),
    ('transformers',                   'Transformers'),
    ('safetensors',                    'Safetensors'),
    ('accelerate',                     'Accelerate'),
    ('einops',                         'Einops'),
    ('kornia',                         'Kornia'),
    ('omegaconf',                      'OmegaConf'),
    ('ldm.models.diffusion.ddpm',      'LDM (SD 1.5/2.1 backend)'),
    ('sgm.modules.encoders.modules',   'SGM (SDXL generative-models)'),
    ('k_diffusion',                    'k-diffusion samplers')
]

print('Vérification des modules individuels...')
ok, fail = 0, 0
for mod, label in modules_a_tester:
    try:
        importlib.import_module(mod)
        print(f'  \u2713 {label}')
        ok += 1
    except Exception as e:
        print(f'  \u274c {label} → ERREUR : {e}')
        fail += 1

# Test complet de la séquence d\'initialisation d\'A1111
print('\nTest de initialize.imports()...')
try:
    from modules import initialize
    initialize.imports()
    print('\u2713✓ initialize.imports() SUCCÈS TOTAL ! Le backend A1111 est 100% opérationnel.')
except Exception as e:
    print(f'\u274c initialize.imports() a échoué : {e}')
    fail += 1

if fail > 0:
    print('\n\u26a0\ufe0f Des dépendances manquent encore. Vérifiez ci-dessus.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 4 — TÉLÉCHARGEMENT DES MODÈLES (Animagine XL 3.1 + SDXL VAE)
# ══════════════════════════════════════════════════
import os, shutil
WEBUI = '/kaggle/working/stable-diffusion-webui'
ckpt_dir = f'{WEBUI}/models/Stable-diffusion'
vae_dir  = f'{WEBUI}/models/VAE'

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

if not os.path.exists(f'{ckpt_dir}/animagine-xl-3.1.safetensors'):
    print('Téléchargement Animagine XL 3.1 (6.4 Go)...')
    !wget -q --show-progress -c https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -O {ckpt_dir}/animagine-xl-3.1.safetensors
else:
    print('✓ Animagine XL 3.1 déjà présent')

if not os.path.exists(f'{vae_dir}/sdxl_vae.safetensors'):
    print('Téléchargement SDXL VAE (319 Mo)...')
    !wget -q --show-progress -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -O {vae_dir}/sdxl_vae.safetensors
else:
    print('✓ SDXL VAE déjà présent')

total, used, free = shutil.disk_usage('/kaggle/working')
print(f'\n✓ Modèles prêts ! Espace disque restant : {free / (1024**3):.1f} Go libres sur {total / (1024**3):.1f} Go')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 5 — TUNNEL CLOUDFLARE + LANCEMENT API DIRECT
# ══════════════════════════════════════════════════
import os, time, re, requests

WEBUI = '/kaggle/working/stable-diffusion-webui'

# Optionnel : Entrez l'URL de votre application AbiArt sur Railway pour auto-connexion !
ABIART_APP_URL = "https://web-production-eebf0.up.railway.app"

# 1. Installer Cloudflare Tunnel dans /tmp (sans encombrer le disque de travail)
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1
!rm -f /tmp/cloudflared.deb

# 2. Démarrer Cloudflare en arrière-plan indépendant
!pkill -f cloudflared 2>/dev/null || true
!rm -f /tmp/tunnel.log
!nohup cloudflared tunnel --url http://127.0.0.1:7860 > /tmp/tunnel.log 2>&1 &

# 3. Attente propre de l'URL Cloudflare dans le log
print('Démarrage du tunnel Cloudflare...')
url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/tmp/tunnel.log'):
        with open('/tmp/tunnel.log', 'r') as f:
            content = f.read()
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if m:
                url = m.group(0)
                break

if url:
    print('\n' + '='*65)
    print('🎨 BACKEND KAGGLE ACTIF :')
    print(f'   >>>  {url}  <<<')
    print('='*65)
    
    if ABIART_APP_URL and ABIART_APP_URL.strip() and 'votre-app' not in ABIART_APP_URL:
        try:
            r = requests.post(f"{ABIART_APP_URL.rstrip('/')}/api/kaggle/register", json={'url': url}, timeout=8)
            if r.status_code == 200:
                print('✅ CONNECTÉ AUTOMATIQUEMENT À ABIART !')
            else:
                print(f'⚠️ Enregistrement auto : HTTP {r.status_code}')
        except Exception as e:
            print(f'⚠️ Note auto-connect : {e}')
    else:
        print('Pour l\'utiliser dans AbiArt : définissez KAGGLE_URL dans Railway.')
else:
    print('⚠️ L\'URL Cloudflare met un peu plus de temps à apparaître, démarrage en cours...')

# 4. Lancement direct d'Automatic1111 (SDPA natif PyTorch 2.x, ultra-léger en VRAM)
%cd {WEBUI}
!python launch.py \
    --skip-prepare-environment \
    --skip-install \
    --skip-torch-cuda-test \
    --skip-python-version-check \
    --enable-insecure-extension-access \
    --opt-sdp-attention \
    --disable-nan-check \
    --api \
    --listen \
    --port 7860